In [1]:
import torch
import math

In [2]:
a = torch.tensor([[[0.2745, 0.6584, 0.2775, 0.8573], #A
[0.8993, 0.0390, 0.9268, 0.7388],
[0.7179, 0.7058, 0.9156, 0.4340]],
[[0.0772, 0.3565, 0.1479, 0.5331],
[0.4066, 0.2318, 0.4545, 0.9737],
[0.4606, 0.5159, 0.4220, 0.5786]]])

print(a.shape)

torch.Size([2, 3, 4])


In [3]:
d_in, d_out, context_length, dropout, num_heads = a.shape[-1], a.shape[-1], a.shape[-2], 0.0, 2

In [4]:
print('d_in:', d_in, '\n', 'd_out:', d_out, '\n', 'context_length:', context_length, '\n', 'num_heads:', num_heads)

d_in: 4 
 d_out: 4 
 context_length: 3 
 num_heads: 2


In [5]:
head_dim = d_out//num_heads
print(head_dim)

2


In [6]:
w_query = torch.torch.nn.Linear(d_in, d_out)
w_key = torch.torch.nn.Linear(d_in, d_out)
w_value = torch.torch.nn.Linear(d_in, d_out)
dropout = torch.torch.nn.Dropout(dropout)

out_proj = torch.torch.nn.Linear(d_out, d_out)



In [7]:
b, num_tokens, d_in = a.shape
print(b, num_tokens, d_in)

2 3 4


In [8]:
query = w_query(a)
key = w_key(a)
value = w_value(a)


In [9]:
print(query.shape, key.shape, value.shape)
print(a.shape)

torch.Size([2, 3, 4]) torch.Size([2, 3, 4]) torch.Size([2, 3, 4])
torch.Size([2, 3, 4])


In [10]:
key = key.view(b, num_tokens, num_heads, head_dim)
value = value.view(b, num_tokens, num_heads, head_dim)
query = query.view(b, num_tokens, num_heads, head_dim)

print('key shape:', key.shape)
print('value shape:', value.shape)
print('query shape:', query.shape)

key shape: torch.Size([2, 3, 2, 2])
value shape: torch.Size([2, 3, 2, 2])
query shape: torch.Size([2, 3, 2, 2])


In [11]:
#transposing for matrix multiplication
key = key.transpose(1,2)
value = value.transpose(1,2)
query = query.transpose(1,2)

print('key shape:', key.shape)
print('value shape:', value.shape)
print('query shape:', query.shape)


key shape: torch.Size([2, 2, 3, 2])
value shape: torch.Size([2, 2, 3, 2])
query shape: torch.Size([2, 2, 3, 2])


In [12]:
key.transpose(2, 3).shape

torch.Size([2, 2, 2, 3])

In [13]:
#calculate attention score by multiplying query and key
attention_score = query @ key.transpose(2, 3)
print(attention_score.shape)

torch.Size([2, 2, 3, 3])


In [14]:
mask = torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
print(mask)
print(mask.shape)

tensor([[0., 1., 1.],
        [0., 0., 1.],
        [0., 0., 0.]])
torch.Size([3, 3])


In [15]:
mask_bool =  mask.bool()[:num_tokens, :num_tokens]
print(mask_bool)

tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])


In [16]:
# masked multi head attention
masked_attention_score = attention_score.masked_fill_(mask_bool, -torch.inf)
print(masked_attention_score)
print(masked_attention_score.shape)

tensor([[[[ 0.2525,    -inf,    -inf],
          [ 0.2121,  0.1208,    -inf],
          [ 0.1592,  0.1090,  0.1425]],

         [[ 0.0855,    -inf,    -inf],
          [-0.0102, -0.0512,    -inf],
          [-0.1736, -0.3437, -0.2694]]],


        [[[ 0.2382,    -inf,    -inf],
          [ 0.2868,  0.2008,    -inf],
          [ 0.2158,  0.1895,  0.1966]],

         [[ 0.0704,    -inf,    -inf],
          [ 0.1073,  0.1462,    -inf],
          [ 0.0069, -0.0170, -0.0061]]]], grad_fn=<MaskedFillBackward0>)
torch.Size([2, 2, 3, 3])


In [17]:
# applying softmax to masked attention scores
masked_attention_weights = torch.softmax(masked_attention_score/key.shape[-1]**0.5, dim=-1)
print(masked_attention_weights)


tensor([[[[1.0000, 0.0000, 0.0000],
          [0.5161, 0.4839, 0.0000],
          [0.3386, 0.3268, 0.3346]],

         [[1.0000, 0.0000, 0.0000],
          [0.5072, 0.4928, 0.0000],
          [0.3545, 0.3143, 0.3312]]],


        [[[1.0000, 0.0000, 0.0000],
          [0.5152, 0.4848, 0.0000],
          [0.3369, 0.3307, 0.3324]],

         [[1.0000, 0.0000, 0.0000],
          [0.4931, 0.5069, 0.0000],
          [0.3362, 0.3306, 0.3332]]]], grad_fn=<SoftmaxBackward0>)


In [18]:
#adding dropout layer 
masked_attention_weights = dropout(masked_attention_weights)

In [19]:
#calculating context vector by multiplying the attention weights with the value matrix
context_vec = masked_attention_weights @ value
print(context_vec)
print(context_vec.shape)

tensor([[[[-0.8399, -0.1381],
          [-0.8269, -0.2266],
          [-0.9203, -0.1559]],

         [[-0.2143,  0.1437],
          [-0.2982,  0.1623],
          [-0.3530,  0.0655]]],


        [[[-0.6003, -0.0364],
          [-0.6603, -0.1520],
          [-0.7083, -0.1403]],

         [[-0.0979,  0.2026],
          [-0.1582,  0.2285],
          [-0.1884,  0.1935]]]], grad_fn=<UnsafeViewBackward0>)
torch.Size([2, 2, 3, 2])


In [20]:
# modifying context vec acc to d_out
context_vec  = context_vec.contiguous().view(b, num_tokens, d_out)
print(context_vec, context_vec.shape)


tensor([[[-0.8399, -0.1381, -0.8269, -0.2266],
         [-0.9203, -0.1559, -0.2143,  0.1437],
         [-0.2982,  0.1623, -0.3530,  0.0655]],

        [[-0.6003, -0.0364, -0.6603, -0.1520],
         [-0.7083, -0.1403, -0.0979,  0.2026],
         [-0.1582,  0.2285, -0.1884,  0.1935]]], grad_fn=<ViewBackward0>) torch.Size([2, 3, 4])


In [21]:
contect_vec = out_proj(context_vec)
print(context_vec, context_vec.shape)

tensor([[[-0.8399, -0.1381, -0.8269, -0.2266],
         [-0.9203, -0.1559, -0.2143,  0.1437],
         [-0.2982,  0.1623, -0.3530,  0.0655]],

        [[-0.6003, -0.0364, -0.6603, -0.1520],
         [-0.7083, -0.1403, -0.0979,  0.2026],
         [-0.1582,  0.2285, -0.1884,  0.1935]]], grad_fn=<ViewBackward0>) torch.Size([2, 3, 4])


In [22]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

NameError: name 'nn' is not defined

In [ ]:
multihead = MultiHeadAttention(d_in, 6, 3, 0.0, 2, qkv_bias=False)

In [ ]:
multihead.forward(a)

In [ ]:
# postional embedding
pe = torch.randn(3,4)
print(pe, pe.shape)

In [ ]:
# adding postional embedding to inputs
pea = a + pe
print(pea)


In [ ]:
def normalize_batch(batch: torch.Tensor, eps: float = 1e-8):
    """
    Normalize a batch of matrices (B, M, N).
    Methods: 'minmax' | 'zscore' | 'l2'
    """
    B = batch.shape[0]
    flat = batch.view(B, -1)  # (B, M*N)

    mean = flat.mean(dim=1).view(-1, 1, 1)
    std  = flat.std(dim=1).view(-1, 1, 1)
    return (batch - mean) / (std + eps)

In [ ]:
#normalize_batch(batch)
pea = normalize_batch(pea)

In [ ]:

def Positional_Encoding(batch, seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(seq_len).unsqueeze(1).float()        # (3, 1)
    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
    )
    pe[:, 0::2] = torch.sin(position * div_term)  # even dims
    pe[:, 1::2] = torch.cos(position * div_term)  # odd dims
    return pe.unsqueeze(0).expand(batch, -1, -1)  # (2, 3, 4)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        # 2*4*768
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x
        # 2*4*768

In [ ]:
#intiating class
transformer = Transformer(d_in = 4, d_out= 4, context_length = 3, dropout=0.1, num_heads = 2)

#positional encding vectors
positional_enc_vec = Positional_Encoding(a.shape[-3], a.shape[-2], a.shape[-1])

# adding positional vectors to input :a
print(transformer.forward(a + positional_enc_vec))

In [ ]:
class LLM(torch.nn.Module):
    def __init__(self, d_in, d_out, context_length, no_of_transformer_blocks, dropout, num_heads, vocab_size):
        super().__init__()
        
        self.d_in = d_in
        self.d_out = d_out
        self.context_length = context_length
        self.dropout = dropout
        self.num_heads = num_heads
    
        # transformer blocks
        self.transformer_block = torch.nn.Sequential(
            *[Transformer(self.d_in, self.d_out, self.context_length, self.dropout, self.num_heads) 
              for _ in range(no_of_transformer_blocks)])

        # ✅ Final output layer: projects from d_out → vocab_size
        self.out_head = torch.nn.Linear(d_out, vocab_size, bias=False)
    
    def forward(self, inputs):  
        
        # positional embedding
        self.pos_emb = Positional_Encoding(inputs.shape[0], self.context_length, self.d_in)
        inputs = inputs + self.pos_emb

        # transformer output
        x = self.transformer_block(inputs)

        # layer norm
        x = normalize_batch(x)

        # ✅ Final linear layer → logits over vocabulary
        logits = self.out_head(x)   # shape: (batch, n_tokens, vocab_size)

        return logits

In [ ]:
class LayerNorm(torch.nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = torch.nn.Parameter(torch.ones(emb_dim))
        self.shift = torch.nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

class GELU(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

    
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context

    for _ in range(max_new_tokens):
        
        # Crop current context if it exceeds the supported context size
        idx_cond = idx[:, -context_size:]
        
        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)  # (batch, n_tokens, vocab_size)
        
        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]  

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [ ]:
import tiktoken

def text_to_tokens(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    return torch.tensor(encoded).unsqueeze(0)

def tokens_to_text(tokens, tokenizer):
    flat = tokens.squeeze(0).tolist()
    return tokenizer.decode(flat)

In [ ]:
tokenizer = tiktoken.get_encoding('gpt2')
text = 'The capital of Australia is'

token_ids = generate_text_simple(
    model=GPTModel(),
    idx=text_to_tokens(text, tokenizer),
    max_new_tokens=5,
    context_size=4
)

print(tokens_to_text(token_ids, tokenizer))

In [ ]:
def calc_loss_batch(logits, targets):
    """
    logits : (batch, n_tokens, vocab_size)
    targets: (batch, n_tokens)  — token IDs shifted by 1
    """
    # Flatten for cross-entropy: (batch * n_tokens, vocab_size) vs (batch * n_tokens,)
    loss = torch.nn.functional.cross_entropy(
        logits.view(-1, logits.size(-1)),
        targets.view(-1)
    )
    return loss

In [ ]:
# Use the same tokenizer you already have
text = "The capital of Australia is Canberra. The capital of France is Paris."
token_ids = tokenizer.encode(text, allowed_special={'<|endoftext|>'})

context_size = 3   # must match your LLM's context_length

# Build (input, target) pairs with a sliding window
inputs, targets = [], []
for i in range(0, len(token_ids) - context_size):
    inputs.append(token_ids[i : i + context_size])
    targets.append(token_ids[i + 1 : i + context_size + 1])  # shifted by 1

inputs  = torch.tensor(inputs)   # (N, context_size)
targets = torch.tensor(targets)  # (N, context_size)
print(f"inputs shape: {inputs.shape}, targets shape: {targets.shape}")

In [ ]:
# reading the text data 
with open('the-verdict.txt','r') as book:
    text = book.read()

In [ ]:
#encoding using tokenizer
tokens = tokenizer.encode(text)
print(len(tokens))

In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
gptdataset = GPTDatasetV1(text, tokenizer, 256, 128)

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
# Train/validation ratio
train_ratio = 0.90
split_idx = int(train_ratio * len(text))
train_data = text[:split_idx]
val_data = text[split_idx:]


In [ ]:
torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=256,
    stride=256,
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length = 256,
    stride =  256,
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

In [ ]:
llm = LLM(d_in = 768, d_out = 768, context_length = 256, no_of_transformer_blocks = 12, dropout = 0, num_heads = 12, vocab_size = 256)

In [ ]:
llm.eval();

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
torch.manual_seed(123)
model = GPTModel()
model.eval(); 

In [ ]:
model.to(device);
torch.manual_seed(123)

with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print(f"Training Loss: {train_loss:.4f}")
print(f"Validation Loss: {val_loss:.4f}")

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode
        
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            tokens_seen += input_batch.numel() # Returns the total number of elements (or tokens) in the input_batch.
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0: 
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Print a sample text after each epoch
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [ ]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_tokens(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = tokens_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()

In [ ]:
# Note:
# Uncomment the following code to calculate the execution time
import time
start_time = time.time()

torch.manual_seed(123)
model = GPTModel()
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train_model(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

# Note:
# Uncomment the following code to show the execution time
end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):

    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # New: Filter logits with top_k sampling
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # New: Apply temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:  # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_tokens("Every effort moves you", tokenizer),
    max_new_tokens=5,
    context_size = 256,
    top_k=25,
    temperature=0.7
)

print("Output text:\n", tokens_to_text(token_ids, tokenizer))

# laoding gpt 2 weights

In [ ]:
from gpt_weights import *

In [ ]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

In [ ]:
print("Settings:", settings)
print("Parameter dictionary keys:", params.keys())

In [ ]:
model = GPTModel(d_in=768, context_length=1024, no_of_transformer_blocks= 12, num_heads = 12)


In [ ]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])
    
    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].mha.W_query.weight = assign(
            gpt.trf_blocks[b].mha.W_query.weight, q_w.T)
        gpt.trf_blocks[b].mha.W_key.weight = assign(
            gpt.trf_blocks[b].mha.W_key.weight, k_w.T)
        gpt.trf_blocks[b].mha.W_value.weight = assign(
            gpt.trf_blocks[b].mha.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].mha.W_query.bias = assign(
            gpt.trf_blocks[b].mha.W_query.bias, q_b)
        gpt.trf_blocks[b].mha.W_key.bias = assign(
            gpt.trf_blocks[b].mha.W_key.bias, k_b)
        gpt.trf_blocks[b].mha.W_value.bias = assign(
            gpt.trf_blocks[b].mha.W_value.bias, v_b)

        gpt.trf_blocks[b].mha.out_proj.weight = assign(
            gpt.trf_blocks[b].mha.out_proj.weight, 
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].mha.out_proj.bias = assign(
            gpt.trf_blocks[b].mha.out_proj.bias, 
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight, 
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias, 
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight, 
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias, 
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale, 
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift, 
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale, 
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift, 
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])



In [ ]:
load_weights_into_gpt(model, params)
model.to(device);